# Tool errors and retries

An error handler turns an exception into a message the model can see. A retry repeats execution. A permanent bug remains broken after retries. Place retry inside error handling so its final exception reaches the handler. [Documentation](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error).

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.faults import tool_error_message
from lab.tools import broken_inventory, make_flaky_tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware, ToolRetryMiddleware
from model_config import get_model, run_config
from lab.faults import ScriptedModel, tool_request
from lab.tools import divide
from langchain_core.messages import AIMessage


In [ ]:
model = get_model()
show_source(broken_inventory)
with expect_error(TypeError):
    broken_inventory.invoke({"product": "keyboard"})

### 1. A permanent bug becomes a tool error message

In [ ]:
# Execute
agent = create_agent(
    model=model, tools=[broken_inventory],
    middleware=[ToolErrorMiddleware(tool_error_message), TraceMiddleware("02_bug", pretty=True)],
)
bug_result = agent.invoke({"messages": [("user", prompt("broken"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(bug_result["messages"], pretty=True)

### 2. A transient service recovers on attempt three
The model makes one tool request; middleware makes three execution attempts.

In [ ]:
# Execute
retry_weather, retry_attempts = make_flaky_tool(failures=2)
agent = create_agent(
    model=model, tools=[retry_weather],
    middleware=[
        ToolRetryMiddleware(max_retries=2, initial_delay=0.1, backoff_factor=2, max_delay=0.2, jitter=False, retry_on=(ConnectionError,), tools=[retry_weather]),
        TraceMiddleware("02_retry", pretty=True),
    ],
)
retry_result = agent.invoke({"messages": [("user", "Check wheather in Warsaw and advise how to dress")]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(retry_result["messages"], pretty=True)
print("Actual execution attempts:", retry_attempts)

### 3. Retries exhausted → controlled error
A failing service stays down. The outer error middleware receives the exception after all retry attempts.

In [ ]:
# Execute
weather, attempts = make_flaky_tool(failures=100)
agent = create_agent(
    model=model, tools=[weather],
    middleware=[
        ToolErrorMiddleware(tool_error_message),
        ToolRetryMiddleware(max_retries=2, initial_delay=0.1, jitter=False, on_failure="error"),
        TraceMiddleware("02_exhausted", pretty=True),
    ],
)
exhausted_result = agent.invoke({"messages": [("user", prompt("weather"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(exhausted_result["messages"], pretty=True)
print("Actual execution attempts:", attempts)

### Recover with corrected arguments

An error message lets the model choose a corrected call. The error handler sanitizes selected `ValueError`s and propagates other exception types by returning `None`. The `tools` allowlist limits which tools it handles.

In [ ]:
# Execute
def handle_division(error, request):
    if isinstance(error, ValueError):
        return f"{request.tool_call['name']} needs a nonzero divisor."
    return None

correction_agent = create_agent(
    model=ScriptedModel(responses=[tool_request("divide", a=12, b=0), tool_request("divide", a=12, b=3), AIMessage(content="4")]),
    tools=[divide], middleware=[ToolErrorMiddleware(handle_division, tools=["divide"])],
)
corrected = correction_agent.invoke({"messages": [("user", "Divide twelve by three.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(corrected["messages"], pretty=True)
assert [m.status for m in corrected["messages"] if m.type == "tool"] == ["error", "success"]

### Propagate an unhandled exception

Retries apply only to matching exception types. `on_failure="continue"` returns an error result, `"error"` raises, and a callable returns custom error content. An outer error handler can process the final exception only when the inner retry raises it.

In [ ]:
# Execute
propagation_agent = create_agent(model=ScriptedModel(responses=[tool_request("broken_inventory", product="unknown")]), tools=[broken_inventory], middleware=[ToolErrorMiddleware(handle_division)])
with expect_error(KeyError):
    propagation_agent.invoke({"messages": [("user", "Read missing stock.")]}, config=run_config())
propagation_checked = True

In [ ]:
# Inspect output
print("Unhandled KeyError propagated:", propagation_checked)